In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

Importing the SparkSession Class and the function that lets us use the column function.
- `SparkSession`: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.html
- `pyspark.sql.types`: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/data_types.html

In [ ]:
AWS_ACCESS_KEY_ID = "xxxxxxxxxxxxx"
AWS_SECRET_ACCESS_KEY = "xxxxxxxxxxx"
# AWS_REGION = "eu-central-1"
OUTPUT_PATH = "s3a://cde-spark-demo/demo/orders/"


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 20:18:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Considering we need to interract with `AWS`, passing in the credentials and setting the `source` and `destination` path in `s3`.

In [ ]:
spark = (
    SparkSession.builder.appName("demo")
    .config("spark.hadoop.fs.s3a.input.stream.type", "classic")
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID)
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
    # .config("spark.hadoop.fs.s3a.endpoint.region", AWS_REGION)
    .getOrCreate()
)

This Cell basically create the Spark Application called `RevenueByRegion`
- `SparkSession.builder.appName`: This is where we input the name of your spark application https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.builder.appName.html
- `config`: Used to pass in the AWS credentials, on cloud platforms, this won't be needed as one will likely use Instance Profile.
- `getOrCreate()`: This basically saying check if we have an existing application called `RevenueByRegion`, otherwise create it.

In [4]:
schema = StructType([
    StructField("order_id", IntegerType(), nullable=False),
    StructField("region", StringType(), nullable=False),
    StructField("amount", IntegerType(), nullable=False),
    StructField("status", StringType(), nullable=False),
])

rows = [
    (101, "EU", 200, "PAID"),
    (102, "US", 50, "CANCELLED"),
    (103, "EU", 300, "PAID"),
    (104, "US", 150, "PAID"),
    (105, "EU", 400, "PAID"),
]

orders = spark.createDataFrame(rows, schema)


- Explicitly defining the schema, so that spark doesn't do that for us, which add some overhead.
- We have 5 rows we are converting to spark DataFrame

In [5]:
orders.coalesce(1).write.mode("overwrite").parquet(OUTPUT_PATH)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
                                                                                

- Write the generated DataFrame to s3.

- `coalesce(1)`: This ensure the data has one partition, which generate one file in s3.
- `Mode`: This takes in Overwrite to ensure the file is overwritten always.
- `Parquet`: The file format to write in s3.

In [7]:
spark.stop()

Stopping the spark application!!!!